## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from haystack import Pipeline, Document
from hayhooks import BasePipelineWrapper
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

In [ ]:
from sem_rag.config.settings import Qdrant, LAMEDataset
from sem_rag.storage.qdrant import get_document_store
from sem_rag.retrieval.components import QueryMetadataExtractor, QueryEmbedder, MMRReranker
from sem_rag.retrieval.schemas.lame import METADATA_SCHEMA, EXTRACTION_SYSTEM_PROMPT
from sem_rag.image.encoding import encode_image

In [ ]:
MMR_CANDIDATES = 50
TOP_K = 5

In [ ]:
TEXT_QUERIES = [
    "Find SEM images of samples with needle-like structures.", # Selected on NASICON/1.2
    "Find the images with spots in a hexagonal pattern.", # Selected on FIB Patterning
    "Smooth flat dark membrane with fine homogeneous grain and even brightness", # Selected on new_chips
    "Cluster of big bright sphere-like particles on a black background." # Selected on MoS2_imaging/R231107C
]

TEXT_META_QUERIES = [
    "Find the images with spots in a hexagonal pattern with magnification greater than 50kx.",
    "Find the images with spots in a hexagonal pattern with magnification less than 20kx and with detector MD.",
    "Smooth flat dark membrane with fine homogeneous grain and even brightness with magnification between 11k and 14k", # Magnification selected on the mag of `new_chips`
    "Cluster of big bright sphere-like particles on a black background with detector E-T and dwell time between 4.5 and 15."
]

In [ ]:
IMAGE_QUERIES = [
    "FIB-maintainence__Spots-15_09_25__30kV_10nA__1",
    "lamellas__sample-lamella-prep__Lamella0_FineApproachNeedleDetection_fib",
    "FIB-patterning__trial-2-holes-hexagonal-array-3s-UHR-48kx-30kev-20pA-2_2",
    "BWO_Pasquale_review__220429_D_5kx_surface"
]

IMAGE_META_QUERIES = [
    ("FIB-maintainence__Spots-15_09_25__30kV_10nA__1", "with magnification between 10000 and 20000 and accelerating voltage around 2"),
    ("FIB-patterning__trial-2-holes-hexagonal-array-3s-UHR-48kx-30kev-20pA-2_2", "with E-T detector"),
    ("FIB-patterning__trial-2-holes-hexagonal-array-3s-UHR-48kx-30kev-20pA-2_2", "with magnification between 48k and 49k and and UH RESOLUTION scan mode"),
    ("BWO_Pasquale_review__220429_D_5kx_surface", "with magnification less than 6k")
]

In [ ]:
document_store = get_document_store(Qdrant.COLLECTION_LAME, Qdrant.EMBEDDING_DIM_QWEN)

## Retrieval Setup

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store(Qdrant.COLLECTION_LAME, Qdrant.EMBEDDING_DIM_QWEN)

        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder())
        self.retrieval_pipeline.add_component("metadata_extractor", QueryMetadataExtractor(METADATA_SCHEMA, EXTRACTION_SYSTEM_PROMPT))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=MMR_CANDIDATES, return_embedding=True))
        self.retrieval_pipeline.add_component("mmr", MMRReranker(top_k=TOP_K))

        self.retrieval_pipeline.connect("metadata_extractor.filters", "retriever.filters")
        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")
        self.retrieval_pipeline.connect("query_embedder.embedding", "mmr.query_embedding")
        self.retrieval_pipeline.connect("retriever.documents", "mmr.documents")

    def extract_query(self, messages: list[dict]) -> tuple[str, list[str]]:
        """Extract the latest user message and uploaded image URLs."""
        message = next((item for item in reversed(messages) if item.get("role") == "user"), None)

        if message is None:
            return "", []

        content = message.get("content", "")
        if isinstance(content, str):
            return content, []

        text = ""
        images = []

        for part in content:
            if part.get("type") == "text":
                text += part.get("text", "")

            elif part.get("type") == "image_url":
                url = part.get("image_url", {}).get("url")
                if url:
                    images.append(url)

        return text, images

    def retrieve_documents(self, question: str, images: list[str]) -> list[Document]:
        """Run embedding, metadata filtering, retrieval, and MMR."""
        retrieval_run_args = {
            "metadata_extractor": {"query": question},
            "query_embedder": {"text": question, "image_data_urls": images},
        }

        response = self.retrieval_pipeline.run(retrieval_run_args, include_outputs_from={"retriever"})

        return {
            "no_mmr": response.get("retriever", {}).get("documents", [])[:TOP_K],
            "mmr": response.get("mmr", {}).get("documents", [])
        }

In [ ]:
retriever = PipelineWrapper()
retriever.setup()

In [ ]:
HIDDEN_META = {"image_id", "thumb_b64"}

In [ ]:
def format_meta(meta: dict) -> str:
    lines = []
    for key, value in meta.items():
        if key in HIDDEN_META:
            continue
        if isinstance(value, float):
            value = f"{value:.3f}"
        lines.append(f"• {key}: {value}")
    return "\n".join(lines)

In [ ]:
def draw_panel(ax, document: Document, label: str, color: str) -> None:
    image = np.load(LAMEDataset.CROPS / f"{document.id}.npy")
    ax.imshow(image, cmap="gray")
    ax.set_title(label, fontsize=8, wrap=True, color=color)
    ax.text(0, -0.02, format_meta(document.meta), transform=ax.transAxes, ha="left", va="top", color=color)

In [ ]:
def show_results(query: str, results: dict[str, list[Document]], query_image_id: str | None = None) -> None:
    query_document = None
    if query_image_id is not None:
        query_document = document_store.get_documents_by_id([query_image_id])[0]

    n_cols = max(1, max(len(documents) for documents in results.values())) + (query_document is not None)

    fig = plt.figure(figsize=(4 * n_cols, 6 * len(results)), layout="constrained")
    subfigs = np.atleast_1d(fig.subfigures(len(results), 1))

    for row, (subfig, (method, documents)) in enumerate(zip(subfigs, results.items())):
        subfig.suptitle(method, fontweight="bold")
        axes = list(np.atleast_1d(subfig.subplots(1, n_cols)))

        for ax in axes:
            ax.axis("off")

        if query_document is not None:
            query_ax = axes.pop(0)
            if row == 0:
                draw_panel(query_ax, query_document, f"QUERY\n{query_image_id}", "tab:red")

        for ax, document in zip(axes, documents):
            draw_panel(ax, document, f"{document.id}\nscore: {document.score:.2f}", "black")

    fig.suptitle(query)
    plt.show()

## Text Retrieval

### No Metadata Queries

In [ ]:
for query in TEXT_QUERIES:
    show_results(query, retriever.retrieve_documents(query, []))

### Metadata Queries

In [ ]:
for query in TEXT_META_QUERIES:
    show_results(query, retriever.retrieve_documents(query, []))

## Image Retrieval

### No Metadata Queries

In [ ]:
for image_id in IMAGE_QUERIES:
    data_url = encode_image(LAMEDataset.CROPS / f"{image_id}.npy")
    show_results(image_id, retriever.retrieve_documents("", [data_url]), query_image_id=image_id)

### Metadata Queries

In [ ]:
for image_id, text in IMAGE_META_QUERIES:
    data_url = encode_image(LAMEDataset.CROPS / f"{image_id}.npy")
    show_results(image_id + " " + text, retriever.retrieve_documents(text, [data_url]), query_image_id=image_id)